# Classification de Cellules sur Images Histologiques
## Projet IA & Optimisation — ISEP Paris 2026

**Membre de l'équipe :** Yann Roquigny-Martin

Ce notebook suit le plan du sujet :
1. Exploration des données
2. Extraction de features (homemade, numpy)
3. Entraînement ML (SVM + Random Forest)
4. Phase de test


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    GridSearchCV, cross_val_score,
    cross_val_predict, StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


---
# I. Exploration des données

Les patches proviennent d'images histologiques colorées en H&E :
les noyaux cellulaires apparaissent en **violet** (hématoxyline)
et le tissu conjonctif en **rose clair** (éosine).

On cherche à classer chaque patch parmi 4 types cellulaires :
**Fibroblast, Lymphocyte, Plasma, Tumor**.


In [ ]:
TRAIN_CSV = 'train.csv'
IMG_DIR   = 'Training dataset - images-20260520'

df_train = pd.read_csv(TRAIN_CSV)
n_samples = len(df_train)
print(f'Nombre total de samples : {n_samples}')

label_counts = df_train['Label'].value_counts().sort_index()
print('\nDistribution des labels :')
print(label_counts)
print('\nClasses disponibles :', df_train['Label'].unique())


In [ ]:
# Distribution en barres
fig, ax = plt.subplots(figsize=(7, 4))
label_counts.plot(kind='bar', ax=ax,
                  color='steelblue', edgecolor='white')
ax.set_title('Distribution des labels (jeu train)')
ax.set_xlabel('Type cellulaire')
ax.set_ylabel('Nombre de samples')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

# 3 patches representatifs par classe
labels_unique = sorted(df_train['Label'].unique())
N_ROWS = 3
fig2, axes2 = plt.subplots(
    N_ROWS, len(labels_unique),
    figsize=(len(labels_unique) * 3, N_ROWS * 3)
)
for col, lbl in enumerate(labels_unique):
    samples = df_train[
        df_train['Label'] == lbl
    ].head(N_ROWS)
    for row, (_, s) in enumerate(samples.iterrows()):
        p = os.path.join(IMG_DIR, f"{s['Image']}.png")
        img = np.array(Image.open(p).convert('RGB'))
        axes2[row, col].imshow(img)
        axes2[row, col].axis('off')
        if row == 0:
            axes2[row, col].set_title(lbl, fontsize=12)
fig2.suptitle('Patches representatifs par classe', fontsize=13)
plt.tight_layout(); plt.show()

# Apparence typique par classe
print('\nApparence typique par classe :')
print(' Lymphocyte : petit noyau rond tres sombre, fond clair')
print(' Tumor      : texture dense irrégulière, nombreuses cellules')
print(' Plasma     : cellule allongée sombre, fond clair')
print(' Fibroblast : cellule fusiforme claire, tissu etalé')


---
# II. Extraction de features

## Stratégie de segmentation
On utilise le **seuillage d'Otsu** (implémenté en numpy pur,
sans scipy) sur l'image en niveaux de gris. En coloration H&E,
les cellules sont plus sombres que le fond → masque = `gray < seuil`.

## Features extraites (26 au total)
| Famille | Features |
|---------|----------|
| **Area** | area_ratio, bb_area_ratio, extent |
| **Shape** | circularity, aspect_ratio, norm_perimeter |
| **Intensity** | gray_mean/std/contrast/skew, r/g/b mean+std, |
|  | histogramme 8 bins, cell_mean/std (intra-masque) |

Toutes ces features sont calculées **avec numpy uniquement**.


In [ ]:
def to_grayscale(img_rgb):
    """RGB -> niveaux de gris (luminance, numpy pur)."""
    r = img_rgb[:, :, 0].astype(float)
    g = img_rgb[:, :, 1].astype(float)
    b = img_rgb[:, :, 2].astype(float)
    return 0.299 * r + 0.587 * g + 0.114 * b


def otsu_threshold(gray):
    """
    Seuil optimal d'Otsu (numpy vectorisé).
    Minimise la variance intra-classe <=> maximise la
    variance inter-classes.
    """
    g = np.clip(gray, 0, 255).astype(np.uint8)
    hist, _ = np.histogram(g.ravel(), bins=256, range=(0, 256))
    hist  = hist.astype(float)
    bins  = np.arange(256, dtype=float)
    n     = float(g.size)
    # Sommes cumulées
    w_b    = np.cumsum(hist)          # poids background
    w_f    = n - w_b                  # poids foreground
    sum_b  = np.cumsum(hist * bins)   # somme intensités bg
    sum_tot = sum_b[-1]
    with np.errstate(divide='ignore', invalid='ignore'):
        m_b = np.where(w_b > 0, sum_b / w_b, 0.0)
        m_f = np.where(w_f > 0,
                       (sum_tot - sum_b) / w_f, 0.0)
    var_inter = w_b * w_f * (m_b - m_f) ** 2
    return int(np.argmax(var_inter))


def get_mask(gray):
    """
    Masque binaire de la cellule via seuillage d'Otsu.
    Les cellules etant plus sombres que le fond en H&E :
    foreground = pixels dont l'intensite < seuil.
    """
    thresh = otsu_threshold(gray)
    return gray < thresh


In [ ]:
def area_features(mask):
    """Features de surface a partir du masque binaire."""
    h, w      = mask.shape
    total_px  = h * w
    cell_area = int(mask.sum())
    area_ratio = cell_area / total_px
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if len(rows) > 0 and len(cols) > 0:
        bb_h   = int(rows[-1] - rows[0] + 1)
        bb_w   = int(cols[-1] - cols[0] + 1)
        bb_area = bb_h * bb_w
        extent  = cell_area / max(bb_area, 1)
    else:
        bb_area = total_px
        extent  = 0.0
    return {
        'area_ratio':    area_ratio,
        'bb_area_ratio': bb_area / total_px,
        'extent':        extent,
    }


def shape_features(mask):
    """Features de forme : circularite, aspect ratio, perimetre."""
    cell_area = int(mask.sum())
    if cell_area == 0:
        return {'circularity': 0.0,
                'aspect_ratio': 1.0,
                'norm_perimeter': 0.0}
    # Perimetre vectorise : pixels fg ayant >=1 voisin bg
    pad = np.pad(mask.astype(int), 1, constant_values=0)
    top = pad[:-2, 1:-1]
    bot = pad[2:,  1:-1]
    lft = pad[1:-1, :-2]
    rgt = pad[1:-1, 2:]
    all_fg = (top == 1) & (bot == 1) & (lft == 1) & (rgt == 1)
    perim  = max(int((mask.astype(bool) & ~all_fg).sum()), 1)
    # Circularite (= 1 pour un cercle parfait)
    circ = 4 * np.pi * cell_area / (perim ** 2)
    # Ratio d'aspect de la bounding box (h/w)
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if len(rows) > 1 and len(cols) > 1:
        ar = (rows[-1]-rows[0]+1) / max(cols[-1]-cols[0]+1, 1)
    else:
        ar = 1.0
    return {
        'circularity':    float(circ),
        'aspect_ratio':   float(ar),
        'norm_perimeter': float(perim / np.sqrt(max(cell_area,1))),
    }


def intensity_features(img_rgb, gray, mask):
    """Stats d'intensite globales, par canal et intra-masque."""
    feats = {}
    g = gray.astype(float)
    # Stats globales niveaux de gris
    mu    = float(np.mean(g))
    sigma = max(float(np.std(g)), 1e-6)
    feats['gray_mean']     = mu
    feats['gray_std']      = sigma
    feats['gray_contrast'] = (float(g.max()) - float(g.min())) / 255
    feats['gray_skew'] = float(
        np.mean(((g - mu) / sigma) ** 3)
    )
    # Statistiques par canal RGB
    for i, ch in enumerate(['r', 'g', 'b']):
        c = img_rgb[:, :, i].astype(float)
        feats[f'{ch}_mean'] = float(np.mean(c))
        feats[f'{ch}_std']  = float(np.std(c))
    # Histogramme 8 bins normalise (niveaux de gris)
    hist, _ = np.histogram(g.ravel(), bins=8, range=(0, 256))
    for i, v in enumerate(hist / g.size):
        feats[f'hist_{i}'] = float(v)
    # Intensite intra-masque
    if mask.sum() > 0:
        cell_px = g[mask]
        feats['cell_mean'] = float(np.mean(cell_px))
        feats['cell_std']  = float(np.std(cell_px))
    else:
        feats['cell_mean'] = mu
        feats['cell_std']  = sigma
    return feats


def extract_features(img_path):
    """Extrait toutes les features d'une image."""
    img  = np.array(Image.open(img_path).convert('RGB'))
    gray = to_grayscale(img)
    mask = get_mask(gray)
    feats = {}
    feats.update(area_features(mask))
    feats.update(shape_features(mask))
    feats.update(intensity_features(img, gray, mask))
    return feats


In [ ]:
records = []
for _, row in df_train.iterrows():
    p = os.path.join(IMG_DIR, f"{row['Image']}.png")
    feats = extract_features(p)
    feats['image'] = row['Image']
    feats['label'] = row['Label']
    records.append(feats)

df_feat = pd.DataFrame(records)
# Sauvegarde des features pour reutilisation
df_feat.to_csv('features_train.csv', index=False)

FEATURE_COLS = [
    c for c in df_feat.columns
    if c not in ('image', 'label')
]
print(f'Features extraites : {len(FEATURE_COLS)}')
print(f'Samples            : {len(df_feat)}')
print(f'\nNoms des features :\n{FEATURE_COLS}')
df_feat[FEATURE_COLS].describe().round(3)


In [ ]:
COLORS = {
    'Lymphocyte': '#1f77b4',
    'Tumor':      '#d62728',
    'Plasma':     '#2ca02c',
    'Fibroblast': '#ff7f0e',
}
pairs = [
    ('area_ratio',  'gray_mean'),
    ('circularity', 'norm_perimeter'),
    ('r_mean',      'b_mean'),
    ('cell_mean',   'cell_std'),
]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (fx, fy) in zip(axes, pairs):
    for lbl, color in COLORS.items():
        m = df_feat['label'] == lbl
        ax.scatter(
            df_feat.loc[m, fx],
            df_feat.loc[m, fy],
            c=color, label=lbl, alpha=0.5, s=10
        )
    ax.set_xlabel(fx, fontsize=9)
    ax.set_ylabel(fy, fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle('Scatter plots des features par type cellulaire',
             y=1.02)
plt.tight_layout()
plt.show()


---
# III. Entraînement ML

## Algorithmes choisis
Deux algorithmes vus en cours :
1. **SVM à noyau RBF** : séparateur à marge maximale, très efficace
   en classification multi-classes avec des features normalisées.
   Hyperparamètres : `C` (régularisation) et `gamma` (largeur RBF).
2. **Random Forest** : ensemble de 100-300 arbres de décision.
   Robuste au bruit, fournit les importances des features.
   Hyperparamètres : `n_estimators`, `max_depth`, `min_samples_split`.

## Pipeline de traitement
```
StandardScaler ──► SelectKBest(f_classif) ──► Classifier
```
- **Normalisation** (`StandardScaler`) : indispensable pour SVM,
  car les features ont des échelles très différentes.
- **Sélection** (`SelectKBest`, test ANOVA F) : retenir les `k`
  features les plus discriminantes pour éviter le sur-apprentissage.
- **GridSearchCV** (5-fold stratifié) : exploration exhaustive
  des hyperparamètres tout en préservant la distribution des classes.


In [ ]:
df_feat = pd.read_csv('features_train.csv')
FEATURE_COLS = [
    c for c in df_feat.columns
    if c not in ('image', 'label')
]
X = df_feat[FEATURE_COLS].values
y = df_feat['label'].values

print(f'Matrice de features : {X.shape}')
print(f'Classes             : {np.unique(y)}')

# Validation croisee 5-fold stratifiee
cv = StratifiedKFold(n_splits=5, shuffle=True,
                     random_state=RANDOM_STATE)


In [ ]:
# ── Algorithme 1 : SVM noyau RBF ─────────────────────────────
# C     : compromis marge/erreurs (grand C = moins de regularisation)
# gamma : rayon d'influence des vecteurs supports
pipe_svm = Pipeline([
    ('scaler',   StandardScaler()),
    ('selector', SelectKBest(f_classif)),
    ('svm',      SVC(kernel='rbf',
                     random_state=RANDOM_STATE)),
])
param_grid_svm = {
    'selector__k': [10, 15, 20, 'all'],
    'svm__C':      [0.1, 1, 10, 100],
    'svm__gamma':  ['scale', 'auto', 0.01, 0.1],
}
gs_svm = GridSearchCV(
    pipe_svm, param_grid_svm,
    cv=cv, scoring='accuracy',
    n_jobs=-1, verbose=0
)
gs_svm.fit(X, y)

print('Meilleurs parametres SVM :', gs_svm.best_params_)
print(f'Meilleure accuracy CV    : {gs_svm.best_score_:.4f}')

# Top 5 combinaisons
res_svm = pd.DataFrame(gs_svm.cv_results_)
top_cols = ['param_selector__k', 'param_svm__C',
            'param_svm__gamma',
            'mean_test_score', 'std_test_score']
print('\nTop 5 combinaisons :')
print(
    res_svm[top_cols]
    .sort_values('mean_test_score', ascending=False)
    .head(5)
    .to_string(index=False)
)


In [ ]:
# ── Algorithme 2 : Random Forest ─────────────────────────────
# n_estimators      : nombre d'arbres
# max_depth         : profondeur max (None = arbres complets)
# min_samples_split : min samples pour diviser un noeud
pipe_rf = Pipeline([
    ('scaler',   StandardScaler()),
    ('selector', SelectKBest(f_classif)),
    ('rf',       RandomForestClassifier(
                     random_state=RANDOM_STATE)),
])
param_grid_rf = {
    'selector__k':           [10, 15, 20, 'all'],
    'rf__n_estimators':      [100, 200, 300],
    'rf__max_depth':         [None, 10, 20],
    'rf__min_samples_split': [2, 5],
}
gs_rf = GridSearchCV(
    pipe_rf, param_grid_rf,
    cv=cv, scoring='accuracy',
    n_jobs=-1, verbose=0
)
gs_rf.fit(X, y)

print('Meilleurs parametres RF :', gs_rf.best_params_)
print(f'Meilleure accuracy CV   : {gs_rf.best_score_:.4f}')

# Top 5 combinaisons
res_rf = pd.DataFrame(gs_rf.cv_results_)
top_cols_rf = [
    'param_selector__k', 'param_rf__n_estimators',
    'param_rf__max_depth', 'param_rf__min_samples_split',
    'mean_test_score', 'std_test_score'
]
print('\nTop 5 combinaisons :')
print(
    res_rf[top_cols_rf]
    .sort_values('mean_test_score', ascending=False)
    .head(5)
    .to_string(index=False)
)


In [ ]:
# Validation croisee sur les meilleurs estimateurs
scores_svm = cross_val_score(
    gs_svm.best_estimator_, X, y,
    cv=cv, scoring='accuracy'
)
scores_rf = cross_val_score(
    gs_rf.best_estimator_, X, y,
    cv=cv, scoring='accuracy'
)

print('SVM  scores CV :', scores_svm.round(4))
print(f'     mean ± std : '
      f'{scores_svm.mean():.4f} ± {scores_svm.std():.4f}')
print()
print('RF   scores CV :', scores_rf.round(4))
print(f'     mean ± std : '
      f'{scores_rf.mean():.4f} ± {scores_rf.std():.4f}')

# Graphe comparatif
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(scores_svm))
ax.bar(x - 0.2, scores_svm, 0.4,
       label=f'SVM  (moy={scores_svm.mean():.3f})',
       color='steelblue')
ax.bar(x + 0.2, scores_rf, 0.4,
       label=f'RF   (moy={scores_rf.mean():.3f})',
       color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in x])
ax.set_ylabel('Accuracy')
ax.set_title('Comparaison 5-fold CV : SVM vs Random Forest')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

# Selection du meilleur modele
if scores_svm.mean() >= scores_rf.mean():
    best_model = gs_svm.best_estimator_
    best_name  = 'SVM'
    best_score = scores_svm.mean()
else:
    best_model = gs_rf.best_estimator_
    best_name  = 'Random Forest'
    best_score = scores_rf.mean()

print(f'\n>>> Modele selectionne : {best_name}')
print(f'>>> Accuracy CV        : {best_score:.4f}')


In [ ]:
# Matrice de confusion (predictions croisees sur train)
y_pred = cross_val_predict(
    best_model, X, y, cv=cv
)
labels_ord = sorted(np.unique(y))
cm = confusion_matrix(y, y_pred, labels=labels_ord)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels_ord)))
ax.set_yticks(range(len(labels_ord)))
ax.set_xticklabels(labels_ord, rotation=30, ha='right')
ax.set_yticklabels(labels_ord)
thresh_cm = cm.max() / 2
for i in range(len(labels_ord)):
    for j in range(len(labels_ord)):
        ax.text(
            j, i, str(cm[i, j]),
            ha='center', va='center',
            color='white' if cm[i, j] > thresh_cm else 'black'
        )
ax.set_xlabel('Label predit')
ax.set_ylabel('Label reel')
ax.set_title(f'Matrice de confusion - {best_name} (CV)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(classification_report(y, y_pred))


In [ ]:
# Importance des features (Random Forest)
selector_rf = gs_rf.best_estimator_.named_steps['selector']
rf_clf      = gs_rf.best_estimator_.named_steps['rf']
support     = selector_rf.get_support()
sel_feats   = [f for f, s in zip(FEATURE_COLS, support) if s]
importances = rf_clf.feature_importances_
idx         = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(importances)),
       importances[idx], color='steelblue')
ax.set_xticks(range(len(importances)))
ax.set_xticklabels(
    [sel_feats[i] for i in idx],
    rotation=45, ha='right', fontsize=9
)
ax.set_title('Random Forest — importance des features')
ax.set_ylabel('Mean decrease in impurity')
plt.tight_layout()
plt.show()


---
# IV. Phase de test

La fonction `predict_test` :
1. Charge toutes les images PNG du dossier de test.
2. Extrait exactement les mêmes 26 features que pendant l'entrainement.
3. Applique le meilleur modèle entrainé.
4. Sauvegarde les prédictions dans `test.csv`
   (meme structure que `train.csv` : colonnes `Image` et `Label`).


In [ ]:
def predict_test(test_img_dir, model, feat_cols,
                 output_csv='test.csv'):
    """
    Applique le modele entraine sur le jeu de test.

    Parametres
    ----------
    test_img_dir : str   - dossier contenant les images de test
    model        : Pipeline sklearn entraine
    feat_cols    : list  - noms des features (meme ordre qu'a l'entrainement)
    output_csv   : str   - chemin de sortie CSV
    """
    img_files = sorted(
        [f for f in os.listdir(test_img_dir)
         if f.endswith('.png')],
        key=lambda f: int(os.path.splitext(f)[0])
    )
    img_ids = [os.path.splitext(f)[0] for f in img_files]

    records = []
    for img_id in img_ids:
        p = os.path.join(test_img_dir, f'{img_id}.png')
        records.append(extract_features(p))

    df_test = pd.DataFrame(records)[feat_cols]
    preds   = model.predict(df_test.values)

    df_out = pd.DataFrame({'Image': img_ids, 'Label': preds})
    df_out.to_csv(output_csv, index=False)
    print(f'Predictions sauvegardees dans {output_csv}')
    print(df_out.head(10).to_string(index=False))
    return df_out


In [ ]:
TEST_DIR = 'test dataset - images'

if os.path.exists(TEST_DIR):
    df_predictions = predict_test(
        TEST_DIR, best_model, FEATURE_COLS
    )
else:
    print('Dossier de test non encore disponible.')
    print(f'Placer les images dans : {os.path.abspath(TEST_DIR)}')
    print('Puis relancer cette cellule.')
    print()
    print(f'Modele selectionne : {best_name}')
    print(f'Accuracy CV        : {best_score:.4f}')
